<a href="https://colab.research.google.com/github/zaibjahan200/Deep-Learning-with-Models/blob/main/tweet_classifier/tweet_classifier_using_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-label Tweet Topic Classification

This notebook demonstrates a multi-label tweet topic classification task using a dataset from Hugging Face. The goal is to preprocess tweet data, build a deep learning model (LSTM-based), train it, and evaluate its performance on classifying tweets into various topics.

In [14]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("cardiffnlp/tweet_topic_multi")

In [15]:
ds

DatasetDict({
    test_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 573
    })
    test_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1679
    })
    train_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4585
    })
    train_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 1505
    })
    train_all: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 6090
    })
    validation_2020: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 573
    })
    validation_2021: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 188
    })
    train_random: Dataset({
        features: ['text', 'date', 'label', 'label_name', 'id'],
        num_rows: 4564
    })
    validati

In [16]:
type(ds)

datasets.dataset_dict.DatasetDict

In [17]:
import pandas as pd
train = pd.DataFrame(ds['train_2020'])
display(train.head())

test = pd.DataFrame(ds['test_2020'])
validation = pd.DataFrame(ds['validation_2020'])

,text,date,label,label_name,id
0,The {@Clinton LumberKings@} beat the {@Cedar R...,2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516324419866624
1,I would rather hear Eli Gold announce this Aub...,2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516440690176006
2,"Someone take my phone away, I’m trying to not ...",2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516543387709440
3,"A year ago, Louisville struggled to beat an FC...",2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516620466429953
4,Anyone know why the #Dodgers #Orioles game nex...,2019-09-08,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",[sports],1170516711411310592


In [18]:
train['label_name'][2][0]

'sports'

In [19]:
def remove_cols(df):
  return df.drop(columns=['date','id'])

def labels_array_to_string(df):
  for i in range(0,df.shape[0]):
    if len(df.loc[i,'label_name']) > 0:
      df.loc[i,'label_name'] = df['label_name'][i][0]
    else :
      df.loc[i,'label_name'] = ''
  return df


In [20]:
# !pip install urlextract

In [21]:
import re
from urlextract import URLExtract
extractor = URLExtract()

def format_tweet(tweet):
    # mask web urls
    urls = extractor.find_urls(tweet)
    for url in urls:
        tweet = tweet.replace(url, "{{URL}}")
    # format twitter account
    tweet = re.sub(r"\b(\s*)(@[\S]+)\b", r'\1{\2@}', tweet)
    return tweet

#Data Preprocessing

## Data Preprocessing Overview

This section handles the initial preparation of the raw tweet data. It involves loading the datasets into pandas DataFrames, dropping irrelevant columns like `date` and `id`, and transforming the `label_name` column from an array of labels to a single string representation. Additionally, the `format_tweet` function is applied to clean and standardize tweet text by masking URLs and formatting Twitter account mentions.

In [22]:
train = remove_cols(train)
test = remove_cols(test)
validation = remove_cols(validation)


In [23]:
train = labels_array_to_string(train)
test = labels_array_to_string(test)
validation = labels_array_to_string(validation)

In [24]:
train['text'] = train['text'].apply(format_tweet)
test['text'] = test['text'].apply(format_tweet)
validation['text'] = validation['text'].apply(format_tweet)

In [25]:
train = train[train['label_name'] != '' ]

In [26]:
num_classes = len(train['label_name'].unique())

In [27]:
train.isnull().sum()

,0
text,0
label,0
label_name,0


In [28]:
test.isnull().sum()

,0
text,0
label,0
label_name,0


In [29]:
validation.isnull().sum()

,0
text,0
label,0
label_name,0


## Tokenization

In this step, the `text` data from the tweets is tokenized. Tokenization involves breaking down the raw text into individual words or subwords (tokens). A custom tokenizer is used to extract relevant terms, converting them to lowercase and handling hashtags and mentions. The resulting tokens are stored in a new `tokens` column in the DataFrame.

In [30]:
import re
def tokenizer(text):
  return re.findall(r'[#@]?\w+', text.lower())

In [31]:
train['tokens'] = train['text'].apply(tokenizer)

## Vocabulary Mapping

This section creates a numerical representation for each unique word (token) in the dataset. A vocabulary map (`vocab`) is built where each unique token is assigned a unique integer ID. Special tokens like `<PAD>` and `<UNK>` (unknown) are also included. The `tokens` are then converted into numerical sequences (`sequence`) based on this vocabulary. This step also determines the total vocabulary size (`vocab_size`) and the maximum sequence length (`max_len`) observed in the training data.

In [32]:
def vocab_map(df):
  vocab = {
      '<PAD>': 0,
      '<UNK>': 1
  }

  idx = 2

  for tokens in df["tokens"]:
      for word in tokens:
          if word not in vocab:
              vocab[word] = idx
              idx += 1

  df["sequence"] = df["tokens"].apply(
      lambda tokens: [vocab.get(word, 1) for word in tokens]
  )
  return df, len(vocab), len(df['tokens'].max())

train, vocab_size, max_len= vocab_map(train)

In [33]:
train.head()

,text,label,label_name,tokens,sequence
0,The {@Clinton LumberKings@} beat the {@Cedar R...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[the, @clinton, lumberkings, beat, the, @cedar...","[2, 3, 4, 5, 2, 6, 7, 8, 9, 10, 11, 12, 13, 14..."
1,I would rather hear Eli Gold announce this Aub...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[i, would, rather, hear, eli, gold, announce, ...","[38, 39, 40, 41, 42, 43, 44, 45, 46, 12, 47, 4..."
2,"Someone take my phone away, I’m trying to not ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[someone, take, my, phone, away, i, m, trying,...","[51, 52, 53, 54, 55, 38, 56, 57, 58, 59, 60, 6..."
3,"A year ago, Louisville struggled to beat an FC...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[a, year, ago, louisville, struggled, to, beat...","[21, 67, 68, 69, 70, 58, 5, 71, 72, 73, 74, 75..."
4,Anyone know why the #Dodgers #Orioles game nex...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",sports,"[anyone, know, why, the, #dodgers, #orioles, g...","[99, 100, 101, 2, 102, 103, 12, 104, 105, 106,..."


## Padding

To ensure that all input sequences to the neural network have a uniform length, padding is applied. Sequences shorter than `max_len` are padded with `0` (corresponding to `<PAD>` token), while sequences longer than `max_len` are truncated. This creates a `padded_sequence` column, which is essential for batch processing in deep learning models.

In [34]:
def padding(df):
    padded_dataset = []

    for sequence in df['sequence']:
        if len(sequence) < max_len:
            sequence = sequence + [0] * (max_len - len(sequence))
        else:
            sequence = sequence[:max_len]

        padded_dataset.append(sequence)

    df['padded_sequence'] = padded_dataset
    return df


In [35]:
train = padding(train)

# Embedding with LSTM and Classifier


## Model Building

Here, a sequential deep learning model is constructed using Keras. The model architecture consists of:

1.  **Embedding Layer**: Converts integer-encoded words into dense vectors of fixed size.
2.  **LSTM Layer**: A Long Short-Term Memory recurrent layer, suitable for processing sequential data like text, with dropout for regularization.
3.  **Dense Layers**: Fully connected layers with ReLU activation for feature transformation.
4.  **Dropout Layer**: Further regularization to prevent overfitting.
5.  **Output Dense Layer**: A final dense layer with `sigmoid` activation, appropriate for multi-label classification, where each output neuron corresponds to a class and predicts the probability of that class being present.

The model is compiled with the `adam` optimizer and `binary_crossentropy` loss function, which is standard for multi-label tasks.

In [82]:
from tensorflow.keras import Sequential
import tensorflow.keras as keras
from keras.layers import Dense,Embedding,LSTM,Dropout

def build_model(vocab_size, max_len, num_classes):
    model = Sequential([
        Embedding(vocab_size, 64, input_length=max_len),
        LSTM(64, dropout=0.2, recurrent_dropout=0.2),
        Dense(32, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


In [83]:
model = build_model(vocab_size, max_len, num_classes)
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [84]:
train['padded_sequence'].apply(len).unique()

array([32])

In [85]:
import numpy as np

X_train = np.array(train['padded_sequence'].tolist())

print(X_train.shape)
print(X_train.dtype)
type(train['label'][0])
y_train = np.array(train['label'].tolist())
print(y_train.shape)

(4584, 32)
int64
(4584, 19)


Forming testing and validation data

In [86]:
# Apply tokenizer to test and validation datasets
test['tokens'] = test['text'].apply(tokenizer)
validation['tokens'] = validation['text'].apply(tokenizer)

# Apply vocab_map to test and validation datasets
# Note: vocab_map implicitly uses the 'vocab' built from the training data
test, _, _ = vocab_map(test)
validation, _, _ = vocab_map(validation)

# Apply padding to test and validation datasets
test = padding(test)
validation = padding(validation)

In [87]:
# Convert padded sequences to numpy arrays for test and validation
X_test = np.array(test['padded_sequence'].tolist())
X_valid = np.array(validation['padded_sequence'].tolist())

# Convert labels to numpy arrays for test and validation
y_test = np.array(test['label'].tolist())
y_valid = np.array(validation['label'].tolist())

print('Shape of X_test:', X_test.shape)
print('Shape of y_test:', y_test.shape)
print('Shape of X_valid:', X_valid.shape)
print('Shape of y_valid:', y_valid.shape)

Shape of X_test: (573, 32)
Shape of y_test: (573, 19)
Shape of X_valid: (573, 32)
Shape of y_valid: (573, 19)


## Training and Testing the Model

This section involves training the defined LSTM model and evaluating its performance. Class weights are computed to address potential class imbalance in the training data, ensuring that the model doesn't overemphasize majority classes. The model is then trained using the `X_train` and `y_train` data, with `X_valid` and `y_valid` used for validation during training. After training, the model's performance is assessed on the unseen `X_test` and `y_test` data, and a classification report provides detailed metrics such as precision, recall, and F1-score for each class.

In [89]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y_train_labels = np.argmax(y_train, axis=1)

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)

class_weights = dict(enumerate(weights))

In [90]:
model.fit(X_train, y_train , validation_data=(X_valid,y_valid),epochs = 22,batch_size = 32, verbose =1,class_weight=class_weights )
x = model.evaluate(X_test,y_test)


Epoch 1/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 27s 168ms/step - accuracy: 0.0908 - loss: 0.3264 - val_accuracy: 0.1885 - val_loss: 0.2874
Epoch 2/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 18s 122ms/step - accuracy: 0.1180 - loss: 0.3099 - val_accuracy: 0.1885 - val_loss: 0.2866
Epoch 3/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 17s 116ms/step - accuracy: 0.1263 - loss: 0.3011 - val_accuracy: 0.1885 - val_loss: 0.2870
Epoch 4/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 20s 138ms/step - accuracy: 0.1311 - loss: 0.3017 - val_accuracy: 0.1885 - val_loss: 0.2847
Epoch 5/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 18s 120ms/step - accuracy: 0.1457 - loss: 0.2958 - val_accuracy: 0.1885 - val_loss: 0.2861
Epoch 6/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 21s 124ms/step - accuracy: 0.1449 - loss: 0.2965 - val_accuracy: 0.1885 - val_loss: 0.2872
Epoch 7/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 20s 124ms/step - accuracy: 0.1521 - loss: 0.2922 - val_accuracy: 0.1885 - val_loss: 0.2859
Epoch 8/22
144/144 ━━━━━━━━━━━━━━━━━━━━ 18s 122ms/step - accuracy: 0.1767 - loss: 0

In [91]:
print(f"Error : {x[0]}\nAccuracy : {x[1]*100}")

Error : 0.450611412525177
Accuracy : 17.277486622333527


In [92]:
from sklearn.metrics import classification_report
import numpy as np

y_pred = model.predict(X_test)

y_pred = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

print(classification_report(y_true, y_pred))

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 46ms/step
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        33
           1       0.00      0.00      0.00        24
           2       0.00      0.00      0.00        81
           3       0.00      0.00      0.00        60
           4       0.00      0.00      0.00         5
           5       0.00      0.00      0.00         4
           6       0.20      0.02      0.03        54
           7       0.00      0.00      0.00        14
           8       0.00      0.00      0.00         5
           9       0.07      0.15      0.09        13
          10       0.00      0.00      0.00         7
          11       0.00      0.00      0.00        29
          12       0.26      0.23      0.25       115
          13       0.00      0.00      0.00         5
          14       0.00      0.00      0.00         2
          15       0.00      0.00      0.00        10
          16       0.21      0.64      0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Save the Model

After training and evaluating the model, it's good practice to save it for future use. This allows you to load the trained model later without retraining, for tasks like making new predictions or deploying it.

In [93]:
model.save('tweet_topic_model.keras')